# Module 6 Lab 3: Conversational Chain for Corporate Operations

## Objective
Build a simple **two-step conversational chain** with LangChain. It keeps the conversation history across turns and helps with routine corporate operations tasks.

## Use Case: Routine Corporate Operations
Support staff often handle the same kinds of requests: locked accounts, invoice backlogs, full shared drives, onboarding delays and failed reports.

The chain works in two steps:
1. **Diagnosis**: asks short clarifying questions over several turns
2. **Solution**: reads the whole conversation and writes an action plan

```
User: "Employee cannot log in"
    |
[Diagnosis - multi-turn, keeps history]
  Asks: "Is the account locked?"
  Asks: "Is it one user or many?"
    |
[Solution - reads full history]
  Action plan, risks, fallback, confirmation step
```

## Part 1: Setup
Install the libraries if needed (the versions match `requirements_module6_lab3.txt`). Then load the API key from `.env`.

In [ ]:
# !pip install -r requirements_module6_lab3.txt

import os
import json
from datetime import datetime

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

print("Imports OK")
print("OpenAI API key found:", "Yes" if os.getenv("OPENAI_API_KEY") else "No")

## Part 2: Conversation Memory
The history is a plain Python list of messages. Each turn adds a `HumanMessage` and an `AIMessage`. The list is passed into the prompt on every call, so the model can see earlier turns.

In [ ]:
history = []
history.append(HumanMessage(content="Employee cannot log in this morning"))
history.append(AIMessage(content="Is the account locked after failed attempts?"))

print("Messages in history:", len(history))
print([m.type for m in history])

# Reset for the lab
history = []

## Part 3: System Prompts
Two prompts: one for asking questions, one for writing the plan.

In [ ]:
DIAGNOSIS_SYSTEM_PROMPT = """You are an experienced corporate operations assistant.
You help staff with routine tasks such as IT support, finance operations,
HR onboarding and business reporting.

Ask one or two clarifying questions at a time.
Identify likely causes and judge the urgency.
Use the earlier messages in this conversation."""

SOLUTION_SYSTEM_PROMPT = """You are a senior operations manager who turns issues into clear action plans.
Put the most urgent actions first. Note risks and a fallback option.
Keep the answer easy to follow."""

SOLUTION_REQUEST = """Based on our conversation so far, give a short action plan with:
1. Immediate actions (next 30 minutes)
2. Short-term actions (today or this week)
3. Risks to consider
4. Fallback option
5. How we will confirm it is resolved"""

print("Prompts ready")

## Part 4: Build the Chains
Each chain is `prompt | llm`. The history is supplied when the chain is run.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=os.getenv("OPENAI_API_KEY")
)

diagnosis_prompt = ChatPromptTemplate.from_messages([
    ("system", DIAGNOSIS_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])

solution_prompt = ChatPromptTemplate.from_messages([
    ("system", SOLUTION_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])

diagnosis_chain = diagnosis_prompt | llm
solution_chain = solution_prompt | llm

print("Chains ready")

## Part 5: Helper Function
`ask` sends one message through a chain, then saves both sides to the history.

In [ ]:
def ask(chain, user_input, history):
    response = chain.invoke({"chat_history": history, "user_input": user_input})
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response.content))
    return response.content

print("ask() ready")

## Part 6: Load a Sample Scenario
The sample file contains five corporate operations scenarios. We use the first one: an employee account lockout.

In [ ]:
with open("data/sample_conversations.json", "r", encoding="utf-8") as f:
    samples = json.load(f)["conversations"]

for s in samples:
    print(s["conversation_id"], "-", s["scenario"])

first = samples[0]
user_turns = [t["message"] for t in first["conversation_history"] if t["role"] == "user"]
print("\nUser turns in scenario:", len(user_turns))

## Part 7: Multi-Turn Diagnosis
Send each user turn in order. The model keeps the history between turns.

In [ ]:
history = []

for i, text in enumerate(user_turns, 1):
    print(f"[Turn {i} - USER]\n{text}\n")
    reply = ask(diagnosis_chain, text, history)
    print(f"[Turn {i} - ASSISTANT]\n{reply}\n")
    print(f"Messages in history: {len(history)}\n" + "-" * 60)

## Part 8: Solution Step
The solution chain reads the full history and writes the action plan.

In [ ]:
plan = ask(solution_chain, SOLUTION_REQUEST, history)
print(plan)

## Part 9: Review and Save the Transcript

In [ ]:
for i, msg in enumerate(history, 1):
    role = "User" if msg.type == "human" else "Assistant"
    preview = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"[{i}] {role}: {preview}")

transcript = {
    "scenario": first["scenario"],
    "timestamp": datetime.now().isoformat(),
    "total_messages": len(history),
    "conversation": [
        {"message_id": i, "role": m.type, "content": m.content}
        for i, m in enumerate(history, 1)
    ]
}

with open("conversation_transcript.json", "w", encoding="utf-8") as f:
    json.dump(transcript, f, indent=2, ensure_ascii=False)

print("\nSaved conversation_transcript.json")

## Part 10: Run Another Scenario
Change the index to try the other samples: 0 = account lockout, 1 = invoice backlog, 2 = shared drive, 3 = new hire laptop, 4 = weekly sales report. Only the user turns are sent.

In [ ]:
scenario_index = 1  # change to try another scenario

chosen = samples[scenario_index]
print("SCENARIO:", chosen["scenario"])

history = []
chosen_turns = [t["message"] for t in chosen["conversation_history"] if t["role"] == "user"]

for i, text in enumerate(chosen_turns, 1):
    reply = ask(diagnosis_chain, text, history)
    print(f"\n[Turn {i} - ASSISTANT]\n{reply}")

print("\n[SOLUTION]")
print(ask(solution_chain, SOLUTION_REQUEST, history))

## Part 11: Key Concepts
- **Memory**: a list of `HumanMessage` and `AIMessage` objects that grows each turn.
- **MessagesPlaceholder**: the spot in the prompt where the history is inserted.
- **Two chains, one history**: the solution chain sees everything the diagnosis chain saw.
- **Simple workflow**: diagnosis questions first, then one action plan.

## Part 12: Exercises
1. Write your own routine operations issue (for example, a printer that is offline for a whole floor) and run it through the diagnosis chain.
2. Compare the plan after 2 user turns versus 3 user turns.
3. Change `DIAGNOSIS_SYSTEM_PROMPT` to ask more or fewer questions.
4. Add a third chain that writes a short status update for managers, using the same history.